### Train a simple DQN to play Mario on Google Colab GPU
DQN stands for Deep Q-Network. It is a landmark algorithm in deep reinforcement learning that combines neural networks with classic Q-learning to play video games directly from raw pixels.

A DQN replaces the traditional Q-table with a neural network. You feed the network raw pixels from the game screen, and it outputs the predicted Q-value for every possible action. For example: 
* Input: A frame of Mario running toward a Goomba.
* Output: [Left: 0.1, Right: 0.7, Jump: 0.8] → The agent chooses Jump.

---

While DQN was the bridge that made deep RL practical, sequential games like Mario often benefit from its memory-augmented successors to handle hidden information.

In this notebook, we have trained a memory-augmented double DQN using the cache method in our agent.
* One of the highlights is how we handle this massive memory properly without triggering a memory leak.
* Another highlight is the attachment of Google Drive. This allows us to routinely save the best checkpoint and the most recent checkpoint. Consequently, we can break those hour-long training sessions into manageable time blocks and continue without losing previous efforts.

---

How long does a single training session last in this notebook?
* This notebook's training session is designed to last 10 hours. Upon a crash, the most recent checkpoint will be saved to Google Drive, allowing you to continue from it in the next session.
* We recommend buying 100-hour compute units from Google at a cost of $10. This will allow you to run about nine training sessions.
* We have attached our best checkpoint to this repository. You can continue from it to reduce your training hours and utilize the free compute units Google offers.

---

If you are still not BORED away, here we go:

what can we learn:
* What's the simulator used in this paper to 'play' Mario.
* What's the input and output for the model used here.
* What's the Metrics to evulate this model training.
* How to create gameplay footage to attract more to join this study group.
* For my personal interest, it is a starter to learn PyTorch.

### AI-powered Mario

This project is based on **Train a Mario-playing RL Agent**: [Reference Paper](https://docs.pytorch.org/tutorials/intermediate/mario_rl_tutorial.html)

Authors: [Yuansong Feng](https://github.com/YuansongFeng), [Suraj Subramanian](https://github.com/suraj813), [Howard Wang](https://github.com/hw26), [Steven Guo](https://github.com/GuoYuzhang).  

---

**Welcome!**

This tutorial walks you through the fundamentals of Deep Reinforcement Learning. At the end, you will implement an AI-powered Mario (using [Double Deep Q-Networks](https://arxiv.org/pdf/1509.06461.pdf)) that can play the game by itself.

Although no prior knowledge of RL is necessary for this tutorial, you can familiarize yourself with these RL [concepts](https://spinningup.openai.com/en/latest/spinningup/rl_intro.html), and have this handy [cheatsheet](https://colab.research.google.com/drive/1eN33dPVtdPViiS1njTW_-r-IYCDTFU7N) as your companion. 

The full code is available [here](https://github.com/yuansongFeng/MadMario/).

### Read about the Lineage of this Version

#### The reduced action space

The environment is wrapped with only two actions:

```python
env = JoypadSpace(env, [["right"],
                        ["right", "A"]])
```

**0 = walk right, 1 = jump right.** That's the entire action set — no left, no B (run/fireball), no crouch.

Why: the NES controller yields 256 raw button combinations, and `gym-super-mario-bros` ships `SIMPLE_MOVEMENT` (7 actions) and `COMPLEX_MOVEMENT` (12). Every extra action widens the output layer, multiplies what the ε-greedy policy has to explore, and dilutes the replay buffer with combinations that are never useful for World 1-1. Two actions are enough to clear the level and make the credit assignment problem dramatically easier on a free-tier GPU budget.

**Consequence for checkpoints:** the final layer is `Linear(512, 2)`. A checkpoint from this notebook is **not** loadable into a 7-action agent (the `Mario_AWS` lineage) — it fails with a shape mismatch. Action-space size is part of the checkpoint lineage, not a tunable.


#### What gamma means here

```python
self.gamma = 0.9
```

Gamma (γ) is the **discount factor** in the TD target:

$$TD_t = r + \gamma \cdot Q_{target}(s', \arg\max_a Q_{online}(s', a))$$

It sets how much a future reward is worth relative to an immediate one. A reward *n* steps ahead is scaled by γⁿ, which gives an effective **planning horizon of roughly 1/(1−γ) steps**:

| γ | Horizon | In game time (1 agent step = 4 NES frames) |
| --- | --- | --- |
| **0.90** *(this notebook)* | ~10 steps | ~0.67 s |
| 0.99 | ~100 steps | ~6.7 s |

At γ = 0.9 the agent is a **reactor**: it optimizes for what happens in the next two-thirds of a second. That is enough to jump a Goomba it can see, but not enough to plan a running approach to a wide gap. Diagnostics on this lineage showed eval deaths scattered across 15 distinct zones with no single zone accounting for more than 13% of failures — the signature of an agent that responds well locally but cannot plan. The parallel `Mario_AWS` lineage moved to γ = 0.99 for that reason.

**Gamma is a lineage switch, not a casual knob.** Raising it rescales Q targets by roughly 10×, so a γ = 0.9 checkpoint is meaningless to a γ = 0.99 agent. If you change gamma, start a fresh checkpoint folder.

#### Exploration schedule (a deviation from the reference tutorial)

`exploration_rate_decay = 0.9999975`, `exploration_rate_min = 0.05`.

The PyTorch tutorial's 0.99999975 needs ~9.2 M steps (~39 h at 66 steps/s) just to reach ε = 0.1 — the agent stays >90% random for entire sessions and mean reward never moves. This value decays 10× faster, reaching ε = 0.1 in ~0.9 M steps (~4 h). The floor was lowered from 0.1 to 0.05 because once the policy is decent, 10% random actions is expensive — a single bad jump ends a run. ε is stored in the checkpoint, so the schedule continues seamlessly across resumes.

### Notebook map — which cell does what

Cells are listed in notebook order. The ones you'll actually interact with are in **bold**.

| Section heading | Type | Purpose |
| --- | --- | --- |
| *Setup (Google Colab + T4 GPU)* | **code** | **STEP 1 — pip installs. Restart the session after this.** |
| *(after restart)* | code | STEP 2 — imports, prints NumPy/Torch versions and CUDA availability. |
| *Initialize Environment* | code | Builds `SuperMarioBros-1-1-v0` and applies the **reduced action space**. |
| *Preprocess Environment* | code | `SkipFrame(4)` → grayscale → resize 84×84 → scale to [0,1] → `FrameStack(4)`, giving a `[4, 84, 84]` state. |
| *Act* | code | Epsilon-greedy action selection and the exploration schedule. |
| *Cache and Recall* | code | `RingReplay` — the preallocated uint8 ring buffer (80,000 transitions) plus pinned staging tensors. |
| *Neural Network* | code | `MarioNet`: 3 conv layers → flatten → 2 dense layers, with frozen `target` copy. |
| *TD Estimate & TD Target* | code | Sets **`self.gamma = 0.9`** and defines the DDQN targets. |
| *Updating the model* | code | Adam (`lr=2.5e-4`), `SmoothL1Loss`, `sync_Q_target()`. |
| *Save checkpoint* | code | `save()` (atomic write) and `load()`. |
| *Putting it all together* | code | `learn()` — burn-in of 10k experiences, learn every 3 steps, sync target every 10k. |
| *Logging* | code | `MetricLogger` — appends to `log` and writes the reward/length/loss/Q plots. |
| ***Let's train!*** | **code** | **← THE TRAINING CELL.** |
| ***Record gameplay to Google Drive*** | **code** | **← THE PLAY / RECORDING CELL.** |

### Setup (Google Colab + T4 GPU)

This notebook was written for the 2020-era OpenAI Gym stack. 

To run it on **today's Colab (Python 3.12, NumPy 2.x, a T4 GPU)** two things have to change:
1. **Pin NumPy < 2.** `gym` and the `nes-py` NES emulator are unmaintained and still use removed aliases such as `np.bool8`, which no longer exist in NumPy 2.0.
2. **Pin `gym==0.25.2`.** This keeps the classic 4-tuple `step()` -> `(obs, reward, done, info)` and obs-only `reset()` API used throughout this tutorial. (gym >= 0.26 switches to a 5-tuple API that `nes-py` does not emit, which would otherwise crash inside gym's `TimeLimit` wrapper.)

PyTorch already ships in Colab with CUDA for the T4, so we do **not** reinstall it.

> ⚠️ **Run the install cell, then `Runtime ▸ Restart session`, then run everything below.** Downgrading NumPy only takes effect after the kernel restarts.

In [ ]:
# === STEP 1: install, then RESTART THE RUNTIME (Runtime > Restart session) ===
# PyTorch is preinstalled in Colab with CUDA, so we don't touch it here.
#  - numpy<2          : gym/nes-py predate NumPy 2.0 (they use the removed np.bool8)
#  - gym==0.25.2      : keeps the classic 4-tuple step / obs-only reset API
#  - opencv-python<4.12: an OpenCV build compatible with numpy<2

!pip install -q "numpy<2" "opencv-python<4.12" "gym==0.25.2" "gym-super-mario-bros==7.4.0"

print("\n\u2705 Install finished.")
print("\u26a0\ufe0f  Now click  Runtime > Restart session,  then run the cells below.")
print("   (NumPy was downgraded; the kernel must restart for it to take effect.)")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
,     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 3.2 MB/s eta 0:00:00
,  Preparing metadata (setup.py) ... done
,   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 11.1 MB/s eta 0:00:00
,   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 29.9 MB/s eta 0:00:00
,   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.0 MB/s eta 0:00:00
,   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 53.9 MB/s eta 0:00:00
,  Building wheel for nes-py (setup.py) ... done
,ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
,jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
,opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
,rasterio 1.5.0 requires numpy>=2, but you h

### Before moving on, restart the session!

This is not optional. NumPy was downgraded underneath a running kernel, and the downgrade only takes effect after a restart.

In [ ]:
# === STEP 2: run this AFTER restarting the runtime ===
import os
import copy
import random, datetime
import numpy as np
import cv2

import torch
from torch import nn
from pathlib import Path
from collections import deque

# Gym is an OpenAI toolkit for RL
import gym
from gym.spaces import Box
from gym.wrappers import FrameStack, GrayScaleObservation, TransformObservation

# NES Emulator for OpenAI Gym
from nes_py.wrappers import JoypadSpace

# Super Mario environment for OpenAI Gym
import gym_super_mario_bros

# Silence the (harmless) deprecation chatter from the unmaintained gym package
import warnings; warnings.filterwarnings("ignore")
gym.logger.set_level(gym.logger.ERROR)

print("numpy:", np.__version__, "| torch:", torch.__version__,
      "| CUDA available:", torch.cuda.is_available())

numpy: 1.26.4 | torch: 2.11.0+cu128 | CUDA available: True


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
,Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
,See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


### RL Definitions

**Environment**
The world that an agent interacts with and learns from.

**Action** $a$ :
How the Agent responds to the Environment. The set of all possible Actions is called *action-space*.

**State** $s$ :
The current characteristic of the Environment. The set of all possible States the Environment can be in is called *state-space*.

**Reward** $r$ :
Reward is the key feedback from Environment to Agent. It is what drives the Agent to learn and to change its future action. An aggregation of rewards over multiple time steps is called **Return**.

 **Optimal Action-Value function** $Q^*(s,a)$ :
Gives the expected return if you start in state $s$, take an arbitrary action $a$, and then for each future time step take the action that maximizes returns. $Q$ can be said to stand for the "quality" of the action in a state. We try to approximate this function.


### Initialize Environment
In Mario, the environment consists of tubes, mushrooms and other components.

When Mario makes an action, the environment responds with the changed (next) state, reward and other info.

In [ ]:
# Initialize Super Mario environment
env = gym_super_mario_bros.make("SuperMarioBros-1-1-v0")

# Limit the action-space to
#   0. walk right
#   1. jump right
env = JoypadSpace(
    env,
    [["right"],
    ["right", "A"]]
)

env.reset()
next_state, reward, done, info = env.step(action=0)
print(f"{next_state.shape},\n {reward},\n {done},\n {info}")

(240, 256, 3),
, 0.0,
, False,
, {'coins': 0, 'flag_get': False, 'life': 2, 'score': 0, 'stage': 1, 'status': 'small', 'time': 400, 'world': 1, 'x_pos': 40, 'y_pos': 79}


### Preprocess Environment
Environment data is returned to the agent in `next_state`. As you saw above, each state is represented by a `[3, 240, 256]` size array. Often that is more information than our agent needs; for instance, Mario's actions do not depend on the color of the pipes or the sky!

We use **Wrappers** to preprocess environment data before sending it to the agent.

`GrayScaleObservation` is a common wrapper to transform an RGB image to grayscale; doing so reduces the size of the state representation without losing useful information. Now the size of each state: `[1, 240, 256]`

`ResizeObservation` downsamples each observation into a square image. New size: `[1, 84, 84]`

`SkipFrame` is a custom wrapper that inherits from `gym.Wrapper` and implements the `step()` function. Because consecutive frames don't vary much, we can skip n-intermediate frames without losing much information. The n-th frame aggregates rewards accumulated over each skipped frame.

`FrameStack` is a wrapper that allows us to squash consecutive frames of the environment into a single observation point to feed to our learning model. This way, we can identify if Mario was landing or jumping based on the direction of his movement in the previous several frames.


In [ ]:
class ResizeObservation(gym.ObservationWrapper):
    def __init__(self, env, shape):
        super().__init__(env)
        if isinstance(shape, int):
            self.shape = (shape, shape)
        else:
            self.shape = tuple(shape)

        obs_shape = self.shape + self.observation_space.shape[2:]
        self.observation_space = Box(low=0, high=255, shape=obs_shape, dtype=np.uint8)

    def observation(self, observation):
        observation = cv2.resize(observation, self.shape, interpolation=cv2.INTER_AREA)
        return observation


class SkipFrame(gym.Wrapper):
    def __init__(self, env, skip):
        """Return only every `skip`-th frame"""
        super().__init__(env)
        self._skip = skip

    def step(self, action):
        """Repeat action, and sum reward"""
        total_reward = 0.0
        done = False
        for i in range(self._skip):
            # Accumulate reward and repeat the same action (4-tuple API)
            obs, reward, done, info = self.env.step(action)
            total_reward += reward
            if done:
                break
        return obs, total_reward, done, info


# Apply Wrappers to environment
env = SkipFrame(env, skip=4)
env = GrayScaleObservation(env, keep_dim=False)
env = ResizeObservation(env, shape=84)
env = TransformObservation(env, f=lambda x: x / 255.)
env = FrameStack(env, num_stack=4)

After applying the above wrappers to the environment, the final wrapped state consists of 4 gray-scaled consecutive frames stacked together, as shown above in the image on the left. Each time Mario makes an action, the environment responds with a state of this structure. The structure is represented by a 3-D array of size `[4, 84, 84]`.


![picture](https://drive.google.com/uc?id=1zZU63qsuOKZIOwWt94z6cegOF2SMEmvD)

### Agent
We create a class `Mario` to represent our agent in the game. Mario should be able to:

- **Act** according to the optimal action policy based on the current state (of the environment).

- **Remember** experiences. Experience = (current state, current action, reward, next state). Mario *caches* and later *recalls* his experiences to update his action policy.

- **Learn** a better action policy over time

In [ ]:
class Mario:
    def __init__():
        pass

    def act(self, state):
        """Given a state, choose an epsilon-greedy action"""
        pass

    def cache(self, experience):
        """Add the experience to memory"""
        pass

    def recall(self):
        """Sample experiences from memory"""
        pass

    def learn(self):
        """Update online action value (Q) function with a batch of experiences"""
        pass

In the following sections, we will populate Mario's parameters and define his functions.

#### Act

For any given state, an agent can choose to do the most optimal action (***exploit***) or a random action (***explore***).

Mario randomly explores with a chance of `self.exploration_rate`; when he chooses to exploit, he relies on `MarioNet` (implemented in `Learn` section) to provide the most optimal action.

In [ ]:
class Mario:
  def __init__(self, state_dim, action_dim, save_dir):
    self.state_dim = state_dim
    self.action_dim = action_dim
    self.save_dir = save_dir

    self.use_cuda = torch.cuda.is_available()
    self.device = 'cuda' if self.use_cuda else 'cpu'

    # Mario's DNN to predict the most optimal action - we implement this in the Learn section
    self.net = MarioNet(self.state_dim, self.action_dim).float()
    if self.use_cuda:
      self.net = self.net.to(device='cuda')

    self.exploration_rate = 1
    # Decay tuned for Colab-scale runs: the tutorial's 0.99999975 needs ~9.2M
    # steps (~39 h at ~66 steps/s) just to reach eps=0.1 — the agent stays
    # >90% random for entire sessions and mean reward never moves. This value
    # (10x faster) reaches eps=0.1 in ~0.9M steps (~4 h), and eps is saved in
    # the checkpoint so the schedule continues seamlessly across resumes.
    self.exploration_rate_decay = 0.9999975
    # Floor lowered from 0.1: once the policy is decent, 10% random actions
    # costs a lot (one bad jump ends a run); 5% keeps some exploration while
    # letting the learned policy actually show its skill.
    self.exploration_rate_min = 0.05
    self.curr_step = 0


  def act(self, state):
    """
    Given a state, choose an epsilon-greedy action and update value of step.

    Inputs:
    state(LazyFrame): A single observation of the current state, dimension is (state_dim)
    Outputs:
    action_idx (int): An integer representing which action Mario will perform
    """
    # EXPLORE
    if np.random.rand() < self.exploration_rate:
        action_idx = np.random.randint(self.action_dim)

    # EXPLOIT
    else:
        # no_grad: inference only — don't build an autograd graph (saves GPU memory)
        with torch.no_grad():
            state = torch.tensor(np.asarray(state), dtype=torch.float32, device=self.device)
            state = state.unsqueeze(0)
            action_values = self.net(state, model='online')
            action_idx = torch.argmax(action_values, axis=1).item()

    # decrease exploration_rate
    self.exploration_rate *= self.exploration_rate_decay
    self.exploration_rate = max(self.exploration_rate_min, self.exploration_rate)

    # increment step
    self.curr_step += 1
    return action_idx


#### Cache and Recall

These two functions serve as Mario's "memory" process.

`cache()`: Each time Mario performs an action, he stores the `experience` to his memory. His experience includes the current *state*, *action* performed, *reward* from the action, the *next state*, and whether the game is *done*.

`recall()`: Mario randomly samples a batch of experiences from his memory, and uses that to learn the game.

In [ ]:
class RingReplay:
    """
    Replay memory as PREALLOCATED numpy ring buffers.

    Why not a deque of arrays? A deque allocates small long-lived blocks on
    every step, interleaved with the multi-MB short-lived batch temporaries
    that learning creates every 3 steps. glibc cannot reuse or release memory
    around those interleaved live blocks, so the process heap grows without
    bound even though nothing is "leaked" — observed as RSS climbing ~7 MB per
    learn call up to an 11 GB plateau while the buffer itself held <1 GB.

    Preallocating fixed arrays once means the steady-state training loop does
    ZERO heap allocation for replay storage: cache() is an in-place slot write,
    recall() gathers into reusable batch arrays.
    """
    def __init__(self, capacity, state_shape=(4, 84, 84), batch_size=32):
        self.maxlen = capacity
        self.states      = np.zeros((capacity,) + state_shape, dtype=np.uint8)
        self.next_states = np.zeros((capacity,) + state_shape, dtype=np.uint8)
        self.actions = np.zeros(capacity, dtype=np.int64)
        self.rewards = np.zeros(capacity, dtype=np.float32)
        self.dones   = np.zeros(capacity, dtype=bool)
        # Pre-touch every page now so the FULL footprint is resident from
        # startup: RAM cost is visible immediately (fail fast if it won't fit)
        # and RSS stays perfectly flat for the rest of the run.
        for arr in (self.states, self.next_states):
            arr.fill(0)
        self.idx = 0
        self.size = 0
        # reusable staging arrays for sampled batches (allocated once)
        self._bs  = np.empty((batch_size,) + state_shape, dtype=np.uint8)
        self._bns = np.empty((batch_size,) + state_shape, dtype=np.uint8)
        # reusable float64 scratch for the LazyFrame -> uint8 conversion
        self._scratch = np.empty(state_shape, dtype=np.float64)

    def __len__(self):
        return self.size

    def __getitem__(self, i):
        return (self.states[i], self.next_states[i],
                self.actions[i], self.rewards[i], self.dones[i])

    @property
    def nbytes(self):
        return (self.states.nbytes + self.next_states.nbytes +
                self.actions.nbytes + self.rewards.nbytes + self.dones.nbytes)

    def add(self, state, next_state, action, reward, done):
        # In-place, allocation-free write into the current slot.
        # Env frames arrive scaled to [0,1] (the x/255. wrapper); multiply back
        # into the reusable scratch buffer, then cast into the uint8 slot.
        np.multiply(np.asarray(state), 255.0, out=self._scratch)
        self.states[self.idx] = self._scratch          # float64 -> uint8 cast on assign
        np.multiply(np.asarray(next_state), 255.0, out=self._scratch)
        self.next_states[self.idx] = self._scratch
        self.actions[self.idx] = action
        self.rewards[self.idx] = reward
        self.dones[self.idx] = done
        self.idx = (self.idx + 1) % self.maxlen
        self.size = min(self.size + 1, self.maxlen)

    def sample_into_staging(self, batch_size):
        # Gather into the preallocated staging arrays — no new allocations.
        ind = np.random.randint(0, self.size, size=batch_size)
        np.take(self.states, ind, axis=0, out=self._bs)
        np.take(self.next_states, ind, axis=0, out=self._bns)
        return self._bs, self._bns, self.actions[ind], self.rewards[ind], self.dones[ind]

    def clear(self):
        self.idx = 0
        self.size = 0


class Mario(Mario): # subclassing for continuity
  def __init__(self, state_dim, action_dim, save_dir):
    super().__init__(state_dim, action_dim, save_dir)
    # Batch 64: the T4 is ~95% idle (the CPU emulator is the bottleneck), so
    # doubling the batch is essentially free in wall-clock and yields smoother
    # gradients / a higher replay ratio. Going far larger buys little, since
    # throughput is capped by the environment, not the GPU.
    self.batch_size = 64
    # Capacity 80,000 x ~56.4 KB = ~4.51 GB, allocated up front so the
    # footprint is fixed and known from the very first episode. Doubled from
    # 40k: at >1.5M total steps, a 40k buffer held only the most recent ~2.5%
    # of experience, so rare successes (flag completions) were overwritten
    # within minutes. 80k doubles their retention while total process RAM
    # stays ~6 GB of Colab's ~12.7 GB.
    self.memory = RingReplay(capacity=80000, state_shape=tuple(state_dim),
                             batch_size=self.batch_size)
    # Reusable PINNED staging tensors for fast, allocation-free H2D copies.
    # Only these two small (0.9 MB) tensors are pinned — not the whole buffer.
    self._t_s  = torch.empty((self.batch_size,) + tuple(state_dim),
                             dtype=torch.uint8, pin_memory=self.use_cuda)
    self._t_ns = torch.empty((self.batch_size,) + tuple(state_dim),
                             dtype=torch.uint8, pin_memory=self.use_cuda)


  def cache(self, state, next_state, action, reward, done):
    """
    Store the experience to self.memory (replay buffer)

    Inputs:
    state (LazyFrame),
    next_state (LazyFrame),
    action (int),
    reward (float),
    done(bool))
    """
    self.memory.add(state, next_state, int(action), float(reward), bool(done))


  def recall(self):
    """
    Retrieve a batch of experiences from memory.
    Copies flow through preallocated staging (numpy -> pinned tensor -> GPU);
    the GPU-side tensors are recycled by torch's caching allocator.
    """
    bs, bns, action, reward, done = self.memory.sample_into_staging(self.batch_size)

    self._t_s.copy_(torch.from_numpy(bs))
    self._t_ns.copy_(torch.from_numpy(bns))
    state = self._t_s.to(self.device, non_blocking=True).float().div_(255.0)
    next_state = self._t_ns.to(self.device, non_blocking=True).float().div_(255.0)

    action = torch.tensor(action, dtype=torch.long, device=self.device)
    reward = torch.tensor(reward, dtype=torch.float32, device=self.device)
    done = torch.tensor(done, dtype=torch.bool, device=self.device)

    return state, next_state, action, reward, done


#### Learn

Mario uses the [DDQN algorithm](https://arxiv.org/pdf/1509.06461) under the hood. DDQN uses two ConvNets - $Q_{online}$ and $Q_{target}$ - that independently approximate the optimal action-value function.

In our implementation, we share feature generator `features` across $Q_{online}$ and $Q_{target}$, but maintain separate FC classifiers for each. $\theta_{target}$ (the parameters of $Q_{target}$) is frozen to prevent updation by backprop. Instead, it is periodically synced with $\theta_{online}$ (more on this later).

### Neural Network

In [ ]:
class MarioNet(nn.Module):
  '''mini cnn structure
  input -> (conv2d + relu) x 3 -> flatten -> (dense + relu) x 2 -> output
  '''
  def __init__(self, input_dim, output_dim):
      super().__init__()
      c, h, w = input_dim

      if h != 84:
          raise ValueError(f"Expecting input height: 84, got: {h}")
      if w != 84:
          raise ValueError(f"Expecting input width: 84, got: {w}")

      self.online = nn.Sequential(
          nn.Conv2d(in_channels=c, out_channels=32, kernel_size=8, stride=4),
          nn.ReLU(),
          nn.Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2),
          nn.ReLU(),
          nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1),
          nn.ReLU(),
          nn.Flatten(),
          nn.Linear(3136, 512),
          nn.ReLU(),
          nn.Linear(512, output_dim)
      )

      self.target = copy.deepcopy(self.online)

      # Q_target parameters are frozen.
      for p in self.target.parameters():
          p.requires_grad = False

  def forward(self, input, model):
      if model == 'online':
          return self.online(input)
      elif model == 'target':
          return self.target(input)

#### TD Estimate & TD Target

Two values are involved in learning:

<br/>

**TD Estimate** - the predicted optimal $Q^*$  for a given state $s$

$$
{TD}_e = Q_{online}^*(s,a)
$$
<br/>  


**TD Target** - aggregation of current reward and the estimated $Q^*$ in the next state $s'$

$$
a' = argmax_{a} Q_{online}(s', a)
$$
$$
{TD}_t = r + \gamma Q_{target}^*(s',a')
$$
<br/>

Because we don’t know what next action $a'$ will be, we use the action $a'$ maximizes $Q_{online}$ in the next state $s'$.

Notice we use the [@torch.no_grad()](https://pytorch.org/docs/stable/generated/torch.no_grad.html#no-grad) decorator on `td_target()` to disable gradient calculations here (because we don't need to backpropagate on $\theta_{target}$).


In [ ]:
class Mario(Mario):
  def __init__(self, state_dim, action_dim, save_dir):
    super().__init__(state_dim, action_dim, save_dir)
    self.gamma = 0.9

  def td_estimate(self, state, action):
    current_Q = self.net(state, model='online')[np.arange(0, self.batch_size), action] # Q_online(s,a)
    return current_Q

  @torch.no_grad()
  def td_target(self, reward, next_state, done):
    next_state_Q = self.net(next_state, model='online')
    best_action = torch.argmax(next_state_Q, axis=1)
    next_Q = self.net(next_state, model='target')[np.arange(0, self.batch_size), best_action]
    return (reward + (1 - done.float()) * self.gamma * next_Q).float()

#### Updating the model

As Mario samples inputs from his replay buffer, we compute $TD_t$ and $TD_e$ and backpropagate this loss down $Q_{online}$ to update its parameters $\theta_{online}$ ($\alpha$ is the learning rate `lr` passed to the `Adam optimizer`)
$$
\theta_{online} \leftarrow \theta_{online} + \alpha \nabla(TD_e - TD_t)
$$
<br/>
$\theta_{target}$ does not update through backpropagation.
Instead, we periodically copy $\theta_{online}$ to $\theta_{target}$
$$
\theta_{target} \leftarrow \theta_{online}
$$

In [ ]:
class Mario(Mario):
    def __init__(self, state_dim, action_dim, save_dir):
      super().__init__(state_dim, action_dim, save_dir)
      self.optimizer = torch.optim.Adam(self.net.parameters(), lr=0.00025)
      self.loss_fn = torch.nn.SmoothL1Loss()

    def update_Q_online(self, td_estimate, td_target) :
      loss = self.loss_fn(td_estimate, td_target)
      self.optimizer.zero_grad()
      loss.backward()
      self.optimizer.step()
      return loss.item()

    def sync_Q_target(self):
      self.net.target.load_state_dict(self.net.online.state_dict())

### Save checkpoint

In [ ]:
class Mario(Mario):
    CHECKPOINT_NAME = "mario_net.chkpt"
    BEST_CHECKPOINT_NAME = "mario_net_best.chkpt"

    def save(self, episode=None, reason="scheduled", filename=None):
        """Save a full, resumable training checkpoint (atomic write)."""
        name = filename or self.CHECKPOINT_NAME
        save_path = self.save_dir / name
        tmp_path = self.save_dir / (name + ".tmp")
        torch.save(
            dict(
                model=self.net.state_dict(),
                optimizer=self.optimizer.state_dict(),
                exploration_rate=self.exploration_rate,
                curr_step=self.curr_step,
                episode=episode if episode is not None else 0,
                best_eval=getattr(self, "best_eval", float("-inf")),
            ),
            tmp_path,
        )
        # Atomic replace: never leaves a half-written checkpoint if we crash mid-save
        os.replace(tmp_path, save_path)
        print(f"[checkpoint:{reason}] saved to {save_path} "
              f"(episode {episode}, step {self.curr_step}, eps {self.exploration_rate:.4f})")

    def load(self):
        """
        Load the checkpoint if one exists and return the episode to resume from.
        Returns 0 (start from scratch) when no checkpoint is found.
        """
        load_path = self.save_dir / self.CHECKPOINT_NAME
        if not load_path.exists():
            print("No checkpoint found — starting training from scratch.")
            self.best_eval = float("-inf")
            return 0

        ckpt = torch.load(load_path, map_location=self.device)
        self.net.load_state_dict(ckpt["model"])
        if "optimizer" in ckpt:
            self.optimizer.load_state_dict(ckpt["optimizer"])
        self.exploration_rate = ckpt["exploration_rate"]
        self.curr_step = ckpt.get("curr_step", 0)
        self.best_eval = ckpt.get("best_eval", float("-inf"))
        start_episode = ckpt.get("episode", 0)
        print(f"Resumed from {load_path}: episode {start_episode}, "
              f"step {self.curr_step}, eps {self.exploration_rate:.4f}, "
              f"best eval {self.best_eval:.0f}")
        return start_episode


### Putting it all together


In [ ]:
class Mario(Mario):
    def __init__(self, state_dim, action_dim, save_dir):
        super().__init__(state_dim, action_dim, save_dir)
        self.burnin = 1e4  # min. experiences in the buffer before training starts
        self.learn_every = 3   # no. of experiences between updates to Q_online
        self.sync_every = 1e4   # no. of experiences between Q_target & Q_online sync


    def learn(self):
      if self.curr_step % self.sync_every == 0:
          self.sync_Q_target()

      # RESUME-SAFETY: gate on how full the buffer actually is, NOT on curr_step.
      # After resuming, curr_step is restored from the checkpoint but the replay
      # buffer starts empty — gating on curr_step would call recall() on an
      # empty/undersized buffer and crash on the very first learn step.
      min_experiences = max(self.batch_size, min(self.burnin, self.memory.maxlen))
      if len(self.memory) < min_experiences:
          return None, None

      if self.curr_step % self.learn_every != 0:
          return None, None

      # Sample from memory
      state, next_state, action, reward, done = self.recall()

      # Get TD Estimate
      td_est = self.td_estimate(state, action)

      # Get TD Target
      td_tgt = self.td_target(reward, next_state, done)

      # Backpropagate loss through Q_online
      loss = self.update_Q_online(td_est, td_tgt)

      return (td_est.mean().item(), loss)


### Logging

In [ ]:
import numpy as np
import time, datetime
import matplotlib.pyplot as plt

class MetricLogger():
    def __init__(self, save_dir):
        self.save_log = save_dir / "log"
        # Append if a log already exists (resumed run); write the header only once.
        if not self.save_log.exists():
            with open(self.save_log, "w") as f:
                f.write(
                    f"{'Episode':>8}{'Step':>8}{'Epsilon':>10}{'MeanReward':>15}"
                    f"{'MeanLength':>15}{'MeanLoss':>15}{'MeanQValue':>15}"
                    f"{'TimeDelta':>15}{'Time':>20}\n"
                )
        self.ep_rewards_plot = save_dir / "reward_plot.jpg"
        self.ep_lengths_plot = save_dir / "length_plot.jpg"
        self.ep_avg_losses_plot = save_dir / "loss_plot.jpg"
        self.ep_avg_qs_plot = save_dir / "q_plot.jpg"

        # History metrics
        self.ep_rewards = []
        self.ep_lengths = []
        self.ep_avg_losses = []
        self.ep_avg_qs = []

        # Moving averages, added for every call to record()
        self.moving_avg_ep_rewards = []
        self.moving_avg_ep_lengths = []
        self.moving_avg_ep_avg_losses = []
        self.moving_avg_ep_avg_qs = []

        # Current episode metric
        self.init_episode()

        # Timing
        self.record_time = time.time()


    def log_step(self, reward, loss, q):
        self.curr_ep_reward += reward
        self.curr_ep_length += 1
        if loss:
            self.curr_ep_loss += loss
            self.curr_ep_q += q
            self.curr_ep_loss_length += 1

    def log_episode(self):
        "Mark end of episode"
        self.ep_rewards.append(self.curr_ep_reward)
        self.ep_lengths.append(self.curr_ep_length)
        if self.curr_ep_loss_length == 0:
            ep_avg_loss = 0
            ep_avg_q = 0
        else:
            ep_avg_loss = np.round(self.curr_ep_loss / self.curr_ep_loss_length, 5)
            ep_avg_q = np.round(self.curr_ep_q / self.curr_ep_loss_length, 5)
        self.ep_avg_losses.append(ep_avg_loss)
        self.ep_avg_qs.append(ep_avg_q)

        self.init_episode()

    def init_episode(self):
        self.curr_ep_reward = 0.0
        self.curr_ep_length = 0
        self.curr_ep_loss = 0.0
        self.curr_ep_q = 0.0
        self.curr_ep_loss_length = 0

    def record(self, episode, epsilon, step):
        mean_ep_reward = np.round(np.mean(self.ep_rewards[-100:]), 3)
        mean_ep_length = np.round(np.mean(self.ep_lengths[-100:]), 3)
        mean_ep_loss = np.round(np.mean(self.ep_avg_losses[-100:]), 3)
        mean_ep_q = np.round(np.mean(self.ep_avg_qs[-100:]), 3)
        self.moving_avg_ep_rewards.append(mean_ep_reward)
        self.moving_avg_ep_lengths.append(mean_ep_length)
        self.moving_avg_ep_avg_losses.append(mean_ep_loss)
        self.moving_avg_ep_avg_qs.append(mean_ep_q)


        last_record_time = self.record_time
        self.record_time = time.time()
        time_since_last_record = np.round(self.record_time - last_record_time, 3)

        print(
            f"Episode {episode} - "
            f"Step {step} - "
            f"Epsilon {epsilon} - "
            f"Mean Reward {mean_ep_reward} - "
            f"Mean Length {mean_ep_length} - "
            f"Mean Loss {mean_ep_loss} - "
            f"Mean Q Value {mean_ep_q} - "
            f"Time Delta {time_since_last_record} - "
            f"Time {datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S')}"
        )

        with open(self.save_log, "a") as f:
            f.write(
                f"{episode:8d}{step:8d}{epsilon:10.3f}"
                f"{mean_ep_reward:15.3f}{mean_ep_length:15.3f}{mean_ep_loss:15.3f}{mean_ep_q:15.3f}"
                f"{time_since_last_record:15.3f}"
                f"{datetime.datetime.now().strftime('%Y-%m-%dT%H:%M:%S'):>20}\n"
            )

        for metric in ["ep_rewards", "ep_lengths", "ep_avg_losses", "ep_avg_qs"]:
            plt.plot(getattr(self, f"moving_avg_{metric}"))
            plt.savefig(getattr(self, f"{metric}_plot"))
            plt.clf()


### Hold up!

If you want to train the DQN agent, run the cell under `Let's train!`.

If you have downloaded the `mario_net_best.chkpt`,  just want to record a gameplay video, run the cell under `Record gameplay to Google Drive`.

### Let's train!

**Mount Google Drive when prompted.**

The training cell calls `drive.mount('/content/drive')` and writes to `/content/drive/MyDrive/mario_rl/checkpoints`. Colab wipes local disk on disconnect, so Drive is what makes a run survive session cutoffs. The auth prompt is interactive — complete it when you start the cell; everything after that runs unattended.

In [ ]:
use_cuda = torch.cuda.is_available()
print(f"Using CUDA: {use_cuda}")
print()

# ---- Checkpoints persist to Google Drive ----
# Colab wipes the local disk when the runtime disconnects, so Drive is what
# makes checkpoints survive session crashes and background-execution cutoffs.
# NOTE: the drive.mount() auth prompt is interactive — complete it when you
# start the cell; after that, background execution proceeds unattended.
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    save_dir = Path('/content/drive/MyDrive/mario_rl/checkpoints')
else:
    # Fixed folder (NOT timestamped) so a re-run of this cell finds the
    # previous checkpoint and resumes instead of starting a fresh directory.
    save_dir = Path('checkpoints')
save_dir.mkdir(parents=True, exist_ok=True)

mario = Mario(state_dim=(4, 84, 84), action_dim=env.action_space.n, save_dir=save_dir)
print(f"Replay buffer preallocated: {mario.memory.nbytes/1e9:.2f} GB "
      f"(fixed for the whole run — RAM use should NOT grow past the first episodes)")

# ---- SANITY GUARD: make sure the ring-buffer cache() is actually in effect ----
# The `class Mario(Mario)` chain means ALL Mario cells must be re-run in order.
# If an old cache() definition is still live in the kernel, memory behavior
# regresses. This catches that immediately instead of an hour in.
_probe = np.zeros((4, 84, 84), dtype=np.float64)
mario.cache(_probe, _probe, 0, 0.0, False)
_s0 = mario.memory[0][0]
if not (isinstance(_s0, np.ndarray) and _s0.dtype == np.uint8 and isinstance(mario.memory, RingReplay)):
    raise RuntimeError(
        "STALE CODE DETECTED: the replay buffer is not the preallocated "
        "uint8 ring buffer.\n"
        "An old version of Mario.cache() is still defined in this kernel.\n"
        "Fix: Runtime > Restart session, then run ALL cells of THIS notebook "
        "in order (do not mix in cells from the old notebook)."
    )
mario.memory.clear()  # remove the probe
print("Buffer check OK: preallocated uint8 ring buffer on CPU.")

# ---- RAM WATCHDOG ----
# When Colab runs out of system RAM, the kernel is KILLED by the OS — Python
# never gets an exception, so a try/except can't save a checkpoint. With the
# preallocated buffer the footprint is fixed, so the watchdog is now a pure
# safety net: it periodically asks glibc to return freed pages to the OS
# (malloc_trim) and, if RAM still gets critical, checkpoints and stops cleanly.
try:
    import psutil  # preinstalled on Colab
    _HAVE_PSUTIL = True
except ImportError:
    _HAVE_PSUTIL = False
    print("psutil not available — RAM watchdog disabled.")

try:
    import ctypes
    _libc = ctypes.CDLL("libc.so.6")
    def malloc_trim():
        _libc.malloc_trim(0)
except Exception:
    def malloc_trim():
        pass

LOW_RAM_GB = 1.5       # below this: malloc_trim + checkpoint, keep going
CRITICAL_RAM_GB = 0.75 # below this: checkpoint and stop cleanly

def available_ram_gb():
    return psutil.virtual_memory().available / 1e9 if _HAVE_PSUTIL else float('inf')

def memory_report():
    if not _HAVE_PSUTIL:
        return ""
    proc_gb = psutil.Process().memory_info().rss / 1e9
    gpu = (f" - GPU {torch.cuda.memory_allocated()/1e9:.2f} GB"
           if torch.cuda.is_available() else "")
    return (f"[mem] process {proc_gb:.2f} GB - buffer {len(mario.memory)}/"
            f"{mario.memory.maxlen} exp ({mario.memory.nbytes/1e9:.2f} GB fixed) "
            f"- free RAM {available_ram_gb():.2f} GB{gpu}")

# 1 & 2: if a checkpoint exists, load it and continue; otherwise start from scratch
start_episode = mario.load()

# ---- EVALUATION (near-greedy, multi-episode) ----
# A PURELY greedy policy in this deterministic env produces one fixed
# trajectory — and it can deadlock: past logs show evals stuck pushing
# against a pipe at x~594 for 2005 steps until the 400 s level timer expired.
# Evaluating with a tiny epsilon (0.02) breaks those deadlocks and, run 3x,
# gives a distribution instead of a single lucky/unlucky rollout. Nothing here
# consumes training steps or touches the replay buffer. The best MEAN eval is
# kept as a separate checkpoint (mario_net_best.chkpt) so a later dip in
# training can never destroy the best policy found.
def greedy_action(state):
    with torch.no_grad():
        s = torch.tensor(np.asarray(state), dtype=torch.float32,
                         device=mario.device).unsqueeze(0)
        return torch.argmax(mario.net(s, model='online'), axis=1).item()

def eval_action(state):
    if np.random.rand() < EVAL_EPSILON:
        return np.random.randint(env.action_space.n)
    return greedy_action(state)

def run_eval_episode(max_steps=3000):
    state = env.reset()
    total_reward, steps, flag, x_pos = 0.0, 0, False, 0
    while steps < max_steps:
        state, reward, done, info = env.step(eval_action(state))
        total_reward += reward
        steps += 1
        x_pos = info.get('x_pos', x_pos)
        if done or info.get('flag_get'):
            flag = bool(info.get('flag_get'))
            break
    return total_reward, steps, flag, x_pos

logger = MetricLogger(save_dir)

# ---- 10-hour Colab Pro background-execution budget ----
# Measured throughput is ~66 steps/s with episodes averaging ~160 steps, so a
# 10 h session covers ~2.4M steps / ~14-15k episodes. The episode target is set
# above that so it never stops the run early; the WALL-CLOCK limit is what ends
# the session: it checkpoints and exits cleanly at 9.5 h, safely before Colab's
# 10 h cutoff would kill the kernel mid-episode with up to ~20 min unsaved.
# Re-run the cell next session to resume exactly where it left off.
episodes = 30000            # ceiling only — the time limit below governs
MAX_TRAIN_HOURS = 9.5       # clean stop + checkpoint before the 10 h cutoff
EVAL_EVERY_EPISODES = 100   # evaluation cadence
EVAL_EPISODES = 3           # episodes per evaluation
EVAL_EPSILON = 0.02         # tiny exploration during eval (see below)
SAVE_EVERY_EPISODES = 300  # 3: checkpoint every 300 episodes

import time
train_start = time.time()

e = start_episode
stopped_early = False
stop_reason = None
train_flags = 0  # times Mario reached the flag during training episodes
try:
    ### for Loop that trains the model by playing the game
    for e in range(start_episode, episodes):

        state = env.reset()

        # Play the game!
        while True:

            # Run agent on the state
            action = mario.act(state)

            # Agent performs action
            next_state, reward, done, info = env.step(action)

            # Remember
            mario.cache(state, next_state, action, reward, done)

            # Learn
            q, loss = mario.learn()

            # Logging
            logger.log_step(reward, loss, q)

            # Update state
            state = next_state

            # Check if end of game
            if done or info['flag_get']:
                if info['flag_get']:
                    train_flags += 1
                break

        logger.log_episode()

        if e % 20 == 0:
            malloc_trim()  # return any freed pages to the OS before measuring
            logger.record(
                episode=e,
                epsilon=mario.exploration_rate,
                step=mario.curr_step
            )
            print(memory_report() + f" - flags reached in training: {train_flags}")

        # ---- Wall-clock budget: stop cleanly before Colab's session cutoff ----
        elapsed_h = (time.time() - train_start) / 3600
        if elapsed_h >= MAX_TRAIN_HOURS:
            print(f"Reached the {MAX_TRAIN_HOURS} h training budget "
                  f"({elapsed_h:.2f} h elapsed) — saving and stopping cleanly.")
            mario.save(episode=e + 1, reason="time-limit")
            stopped_early = True
            stop_reason = "time budget reached"
            break

        # ---- Evaluation + best-model checkpoint ----
        if (e + 1) % EVAL_EVERY_EPISODES == 0:
            results = [run_eval_episode() for _ in range(EVAL_EPISODES)]
            rewards = [r[0] for r in results]
            mean_r, max_r = sum(rewards) / len(rewards), max(rewards)
            max_x = max(r[3] for r in results)
            flags = sum(1 for r in results if r[2])
            print(f"[eval @ ep {e+1}] mean {mean_r:.0f} - max {max_r:.0f} - "
                  f"furthest x_pos {max_x} - flags {flags}/{EVAL_EPISODES} "
                  f"(best mean so far {max(mario.best_eval, mean_r):.0f})")
            if mean_r > mario.best_eval:
                mario.best_eval = mean_r
                mario.save(episode=e + 1, reason="new-best-eval",
                           filename=Mario.BEST_CHECKPOINT_NAME)

        # ---- RAM watchdog: act BEFORE the OS kills the kernel ----
        free_gb = available_ram_gb()
        if free_gb < CRITICAL_RAM_GB:
            print(f"CRITICAL: only {free_gb:.2f} GB RAM free — saving and stopping.")
            mario.save(episode=e + 1, reason="critical-low-ram")
            stopped_early = True
            stop_reason = "critically low RAM"
            break
        elif free_gb < LOW_RAM_GB:
            malloc_trim()
            if available_ram_gb() < LOW_RAM_GB:
                print(f"LOW RAM ({available_ram_gb():.2f} GB free after trim) — "
                      f"checkpointing as a precaution.")
                mario.save(episode=e + 1, reason="low-ram")

        # 3: periodic checkpoint every 300 episodes
        if (e + 1) % SAVE_EVERY_EPISODES == 0:
            mario.save(episode=e + 1, reason="every-300-episodes")

except KeyboardInterrupt:
    # Manual stop (Runtime > Interrupt) — save so the run is resumable
    mario.save(episode=e, reason="keyboard-interrupt")
    raise

except Exception as err:
    # 3: any error (including CUDA OOM) — save before propagating
    print(f"Training hit an error at episode {e}: {err!r}")
    mario.save(episode=e, reason="error")
    raise

else:
    if stopped_early:
        print(f"Stopped early ({stop_reason}). Just re-run this cell to resume "
              "from the checkpoint — training will pick up where it left off.")
    else:
        # Completed all episodes — final checkpoint
        mario.save(episode=episodes, reason="training-complete")


### Record gameplay to Google Drive

Loads the **best** checkpoint (`mario_net_best.chkpt`, falling back to the rolling one), plays up to 5 near-greedy episodes (ε = 0.02, same as evaluation), and saves the best attempt
* a flag run if one happens, otherwise the furthest
* as an H.264 `.mp4` in the Drive checkpoint folder, named like `mario_play_FLAG_20260713_101533.mp4`.

Notes:
- Run the **environment** and **MarioNet** cells first (no need to run training). It never touches training state or the replay buffer, so it's safe to run any time, including right after a training session in the same runtime.
- Frames are the raw 240×256 NES screen captured once per agent step (= 4 emulated frames), so 15 fps plays back at authentic game speed. A full flag run is ~90 seconds of video.
- The video is re-encoded with ffmpeg to H.264/yuv420p so it previews directly in Google Drive and plays in any browser or phone.

In [ ]:
# ---- Record gameplay to an .mp4 on Google Drive ----
# Loads the BEST checkpoint (falls back to the rolling one), plays a few
# near-greedy episodes, records the raw NES screen, and saves the best attempt
# (a flag run if any, otherwise the furthest) as an H.264 .mp4 on Drive.
# Requires the environment + MarioNet cells to have been run first. Safe to run
# any time — it never touches training state or the replay buffer.
import subprocess
import datetime as _dt

# Reuse the training cell's Drive-backed save_dir if it exists; otherwise mount.
if 'save_dir' in globals():
    ckpt_dir = Path(save_dir)
else:
    if not Path('/content/drive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    ckpt_dir = Path('/content/drive/MyDrive/mario_rl/checkpoints')

_best = ckpt_dir / "mario_net_best.chkpt"
_roll = ckpt_dir / "mario_net.chkpt"
ckpt_path = _best if _best.exists() else _roll
assert ckpt_path.exists(), f"No checkpoint found in {ckpt_dir}"

_device = 'cuda' if torch.cuda.is_available() else 'cpu'
play_net = MarioNet((4, 84, 84), env.action_space.n).float().to(_device)
_ckpt = torch.load(ckpt_path, map_location=_device)
play_net.load_state_dict(_ckpt['model'])
play_net.eval()
print(f"Loaded {ckpt_path.name} (episode {_ckpt.get('episode', '?')}, "
      f"best eval {_ckpt.get('best_eval', float('nan')):.0f})")

PLAY_EPSILON = 0.02   # same tiny noise as evaluation: breaks pipe deadlocks
PLAY_ATTEMPTS = 5     # record up to this many episodes, keep the best one
PLAY_MAX_STEPS = 3000
PLAY_FPS = 15         # one recorded frame per agent step = 4 NES frames = real-time speed


def _play_action(state):
    if np.random.rand() < PLAY_EPSILON:
        return np.random.randint(env.action_space.n)
    with torch.no_grad():
        s = torch.tensor(np.asarray(state), dtype=torch.float32,
                         device=_device).unsqueeze(0)
        return torch.argmax(play_net(s, model='online'), axis=1).item()


def _record_episode():
    state = env.reset()
    frames = [env.unwrapped.screen.copy()]
    total_reward, flag, x_pos = 0.0, False, 0
    for _ in range(PLAY_MAX_STEPS):
        state, reward, done, info = env.step(_play_action(state))
        frames.append(env.unwrapped.screen.copy())  # raw 240x256x3 RGB NES frame
        total_reward += reward
        x_pos = info.get('x_pos', x_pos)
        if done or info.get('flag_get'):
            flag = bool(info.get('flag_get'))
            break
    return frames, total_reward, flag, x_pos


best_run = None
for attempt in range(1, PLAY_ATTEMPTS + 1):
    frames, r, flag, x = _record_episode()
    print(f"attempt {attempt}: reward {r:.0f} - x_pos {x} - "
          f"{'FLAG!' if flag else 'no flag'} - {len(frames)} frames")
    if best_run is None or (flag, x) > (best_run[2], best_run[3]):
        best_run = (frames, r, flag, x)
    if flag:
        break  # got a winning run — no need to keep trying

frames, r, flag, x = best_run
_stamp = _dt.datetime.now().strftime("%Y%m%d_%H%M%S")
_label = "FLAG" if flag else f"x{x}"
out_path = ckpt_dir / f"mario_play_{_label}_{_stamp}.mp4"

# Write with cv2 (mp4v), then re-encode to H.264 with Colab's ffmpeg so the
# file previews/plays everywhere (Drive, browsers, phones).
_tmp = Path("/tmp/_mario_play_raw.mp4")
h, w = frames[0].shape[:2]
vw = cv2.VideoWriter(str(_tmp), cv2.VideoWriter_fourcc(*"mp4v"), PLAY_FPS, (w, h))
assert vw.isOpened(), "cv2 VideoWriter failed to open"
for f in frames:
    vw.write(cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
vw.release()

_res = subprocess.run(
    ["ffmpeg", "-y", "-loglevel", "error", "-i", str(_tmp),
     "-vcodec", "libx264", "-pix_fmt", "yuv420p", "-crf", "23", str(out_path)],
    capture_output=True, text=True)
if _res.returncode != 0:
    # ffmpeg unavailable/failed: keep the mp4v file rather than lose the run
    import shutil
    shutil.copy(_tmp, out_path)
    print("ffmpeg re-encode failed — saved mp4v-encoded file instead.")

print(f"\nSaved: {out_path}")
print(f"       {len(frames)} frames @ {PLAY_FPS} fps = "
      f"{len(frames)/PLAY_FPS:.0f} s of gameplay - "
      f"reward {r:.0f} - {'flag reached' if flag else f'reached x_pos {x}'}")
